# Starbucks Capstone Challenge

### Introduction

This data set contains simulated data that mimics customer behavior on the Starbucks rewards mobile app. Once every few days, Starbucks sends out an offer to users of the mobile app. An offer can be merely an advertisement for a drink or an actual offer such as a discount or BOGO (buy one get one free). Some users might not receive any offer during certain weeks. 

Not all users receive the same offer, and that is the challenge to solve with this data set.

Your task is to combine transaction, demographic and offer data to determine which demographic groups respond best to which offer type. This data set is a simplified version of the real Starbucks app because the underlying simulator only has one product whereas Starbucks actually sells dozens of products.

Every offer has a validity period before the offer expires. As an example, a BOGO offer might be valid for only 5 days. You'll see in the data set that informational offers have a validity period even though these ads are merely providing information about a product; for example, if an informational offer has 7 days of validity, you can assume the customer is feeling the influence of the offer for 7 days after receiving the advertisement.

You'll be given transactional data showing user purchases made on the app including the timestamp of purchase and the amount of money spent on a purchase. This transactional data also has a record for each offer that a user receives as well as a record for when a user actually views the offer. There are also records for when a user completes an offer. 

Keep in mind as well that someone using the app might make a purchase through the app without having received an offer or seen an offer.

### Importing libraries and loading datasets

In [ ]:
import pandas as pd
import numpy as np
import math
import json
%matplotlib inline

In [ ]:
# read in the json files
portfolio = pd.read_json('data/portfolio.json', orient='records', lines=True)
profile = pd.read_json('data/profile.json', orient='records', lines=True)
transcript = pd.read_json('data/transcript.json', orient='records', lines=True)

**Importing additional libraries**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

**Showing the first to rows of the datasets**

In [ ]:
portfolio.head(2)

In [ ]:
profile.head(2)

In [ ]:
transcript.head(2)

### Data accessing and cleansing

#### 1. Portfolio
Dataset overview: **portfolio.json**
* id (string) - offer id
* offer_type (string) - type of offer ie BOGO, discount, informational
* difficulty (int) - minimum required spend to complete an offer
* reward (int) - reward given for completing an offer
* duration (int) - time for offer to be open, in days
* channels (list of strings)

##### 1.1 Data Accessing

In [ ]:
portfolio.head()

In [ ]:
portfolio.shape

In [ ]:
portfolio.info()

In [ ]:
portfolio.describe()

In [ ]:
set(portfolio['offer_type'].unique())

In [ ]:
portfolio['channels']

##### 1.2 Data Cleansing
* Splitting the 'channels' column into columns for each fo the channels
* Rename the column 'id' to 'offer_id'

In [ ]:
portfolio_copy = portfolio.copy()

In [ ]:
dummy = pd.get_dummies(portfolio_copy.channels.apply(pd.Series).stack()).sum(level=0)
portfolio_copy = pd.concat([portfolio_copy, dummy], axis=1)
portfolio_copy = portfolio_copy.drop(columns='channels')

In [ ]:
portfolio_copy = portfolio_copy.rename(columns={'id':'offer_id'})

In [ ]:
portfolio_copy

#### 2. Profile
Dataset overview: **profile.json**
* age (int) - age of the customer 
* became_member_on (int) - date when customer created an app account
* gender (str) - gender of the customer (note some entries contain 'O' for other rather than M or F)
* id (str) - customer id
* income (float) - customer's income


##### 2.1 Data Accessing

In [ ]:
profile.head()

In [ ]:
profile.shape

In [ ]:
profile.info()

In [ ]:
profile.describe()

In [ ]:
set(profile['gender'].unique())

##### 2.2 Data Cleansing

* Converting the column 'became_member_on' to a date format
* Renaming the column 'id' to 'customer_id' to clarify the meaning behind it

In [ ]:
profile_copy = profile.copy()

In [ ]:
profile_copy['became_member_on'] = pd.to_datetime(profile_copy['became_member_on'], format='%Y%m%d')

In [ ]:
profile_copy = profile_copy.rename(columns={'id':'customer_id'})

In [ ]:
profile_copy.head()

In [ ]:
profile_copy.info()

#### 3. Transcript
Dataset overview: **transcript.json**
* event (str) - record description (ie transaction, offer received, offer viewed, etc.)
* person (str) - customer id
* time (int) - time in hours since start of test. The data begins at time t=0
* value - (dict of strings) - either an offer id or transaction amount depending on the record

##### 3.1 Data Accessing

In [ ]:
transcript.head()

In [ ]:
transcript.shape

In [ ]:
transcript.info()

In [ ]:
transcript.describe()

In [ ]:
transcript['event'].value_counts()

In [ ]:
transcript['value']

##### 3.2 Data cleansing

* Renaming the column 'person' to 'customer_id' to match the previous dataframe
* Convert the column 'event' into four different columns with dummy values
* Splitting the column 'value' into two columns

In [ ]:
transcript_copy = transcript.copy()

In [ ]:
transcript_copy = transcript_copy.rename(columns={'person':'customer_id'})

In [ ]:
transcript_copy['event'] = transcript_copy['event'].str.replace(' ', '-')
transcript_copy['event'].value_counts()

In [ ]:
dummy = pd.get_dummies(transcript_copy['event'])
transcript_copy = pd.concat([transcript_copy, dummy], axis=1 )

In [ ]:
transcript_copy

In [ ]:
transcript_copy['offer_id'] = [[*i.values()][0]if [*i.keys()][0] in ['offer id','offer_id'] else None for i in transcript_copy.value]
transcript_copy['amount'] = [np.round([*i.values()][0], decimals=2)if [*i.keys()][0] == 'amount' else None for i in transcript_copy.value]

In [ ]:
transcript_copy.head(2)

In [ ]:
transcript_copy = transcript_copy.drop(columns='value')

In [ ]:
transcript_copy.head()

#### Merging all three datasets into one big dataset

In [ ]:
tran_prof = pd.merge(transcript_copy, profile_copy, on='customer_id')

In [ ]:
tran_prof.head(3)

In [ ]:
df = pd.merge(tran_prof, portfolio_copy, on='offer_id', how='left')
df.head(3)

In [ ]:
offer_id = df['offer_id'].unique()
offer_id

In [ ]:
offer_dict = dict([(value, key) for key, value in pd.Series(offer_id ).to_dict().items()]) 
offer_dict

In [ ]:
df['offer_id'] = df['offer_id'].map(offer_dict)
df.head()

In [ ]:
df['offer_id'] = df['offer_id'].replace(1, np.nan)
df.head()

In [ ]:
set(df['offer_id'].unique())

In [ ]:
event_ids = df['event'].unique()
event_ids

In [ ]:
event_dict = pd.Series(event_ids).to_dict()
event_dict

In [ ]:
event_dict = dict([(value, key) for key, value in event_dict.items()]) 
event_dict

In [ ]:
df['event_id'] = df['event'].map(event_dict)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.to_csv('data/data.csv', index=False)

In [ ]:
data = pd.read_csv('data/data.csv')

In [ ]:
data.head(3)

### Data exploration and visualization

In [ ]:
data['age'].describe()

In [ ]:
data['age'].hist(bins = 30)
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.title('Age Group Distribution');

##### Detection
* The dataset contains a lot of outliers with age > 115. This doesn't make sense at all.

In [ ]:
data['income'].describe()

In [ ]:
data['income'].hist(bins = 30);
plt.xlabel('Income Range')
plt.ylabel('Count')
plt.title('Income Range Distribution');

##### Detection
* The highest customer count with > 16,000 is in the income gorup between 70,000 and 75,000

In [ ]:
data['gender'].value_counts()

In [ ]:
males  = data['gender'].value_counts()[0] / data.shape[0]*100
females = data['gender'].value_counts()[1] / data.shape[0]*100
others = data['gender'].value_counts()[2] / data.shape[0]*100

males, females, others

In [ ]:
ax = data['gender'].value_counts()
ax.plot(kind='bar')
plt.ylabel('Number of People')
plt.xlabel('Gender')
plt.title('Gender Distribution');

##### Detection
* The datasets contains more males than females or others

In [ ]:
offer_received = data[data['offer-received'] == 1]['offer_type'].value_counts()
offer_viewed = data[data['offer-viewed'] == 1]['offer_type'].value_counts()
offer_completed = data[data['offer-completed'] == 1]['offer_type'].value_counts()

offer_received, offer_viewed, offer_completed 

In [ ]:
plt.subplot(131)
offer_received = data[data['offer-received'] == 1]['offer_type'].value_counts()
offer_received.plot(kind='bar', figsize=(15,5))
plt.ylabel('counts')
plt.xlabel('Offer Type')
plt.title('Offer received with Offer Type');

plt.subplot(132)
offer_viewed = data[data['offer-viewed'] == 1]['offer_type'].value_counts()
offer_viewed.plot(kind='bar' , figsize=(15,5))
plt.ylabel('counts')
plt.xlabel('Offer Type')
plt.title('Offer viewed with Offer Type');

plt.subplot(133)
offer_completed = data[data['offer-completed'] == 1]['offer_type'].value_counts()
offer_completed.plot(kind='bar' , figsize=(15,5))
plt.ylabel('counts')
plt.xlabel('Offer Type')
plt.title('Offer completed received with Offer Type');

In [ ]:
BOGO_R = offer_received[1] 
BOGO_V = offer_viewed[0] 
BOGO_C = offer_completed[1] 

view_prop = BOGO_V/BOGO_R
com_prop = BOGO_C/BOGO_R
BOGO_R, BOGO_V, BOGO_C, round(view_prop * 100, 2), round(com_prop * 100, 2)

In [ ]:
DISCOUNT_R = offer_received[0] 
DISCOUNT_V = offer_viewed[1] 
DISCOUNT_C = offer_completed[0] 

view_prop = DISCOUNT_V/DISCOUNT_R
com_prop = DISCOUNT_C/DISCOUNT_R
DISCOUNT_R, DISCOUNT_V, DISCOUNT_C, round(view_prop * 100, 2), round(com_prop * 100, 2)

##### Detection
* The most completed offers were for the offer type 'discount'
* The number of offers received were the highest for the discounts with 30,543, while bogo had 30,499 offers.
* The view_prop-variable above for 'discount' shows that ~59% of the received offers were actually completed, while ~70% of the received offers were viewed.
* The view_prop-variable above for 'bogo' shows that ~51% of the received offers were actually completed, while ~83% of the received offers were viewed.

In [ ]:
offer_received = data[data['offer-received'] == 1]['offer_id'].value_counts()
offer_viewed = data[data['offer-viewed'] == 1]['offer_id'].value_counts()
offer_completed = data[data['offer-completed'] == 1]['offer_id'].value_counts()

offer_received, offer_viewed, offer_completed 

In [ ]:
plt.subplot(131)
offer_received = data[data['offer-received'] == 1]['offer_id'].value_counts()
offer_received.plot(kind='bar', figsize=(15,5))
plt.ylabel('counts')
plt.xlabel('Offer Id ')
plt.title('Offer received with Offer Id');

plt.subplot(132)
offer_viewed = data[data['offer-viewed'] == 1]['offer_id'].value_counts()
offer_viewed.plot(kind='bar' , figsize=(15,5))
plt.ylabel('counts')
plt.xlabel('Offer Id')
plt.title('Offer viewed with Offer Id');

plt.subplot(133)
offer_completed = data[data['offer-completed'] == 1]['offer_id'].value_counts()
offer_completed.plot(kind='bar' , figsize=(15,5))
plt.ylabel('counts')
plt.xlabel('Offer Id')
plt.title('Offer completed received with Offer Id');

##### Detection
* Every Offer Id received almost the same amount of offers.
* The viewed offers however differ a lot. Offer Id 10, 8, 9 and 4 are almost at the same level, the lowest viewed count is for the offer id 5
* The most completed offers however were for Offer Id 8, followed by 10 with a little difference. The numbers can be seen three steps above.

In [ ]:
data[data['offer_type']=='bogo'].groupby('customer_id')['offer-received'].count()

In [ ]:
data[data['offer_type']=='bogo'].groupby('customer_id')['offer-received'].count().hist();
plt.title('BOGO Offer Received by User');

##### Detection
* The BOGO offers were received by a lot of users

In [ ]:
data[data['offer_type']=='informational'].groupby('customer_id')['offer-viewed'].count()

In [ ]:
data[data['offer_type']=='informational'].groupby('customer_id')['offer-viewed'].count().hist();
plt.title('Informational Offer Received by User');

##### Detection
* There is a huge difference between the count of the views
* The highest ratio of viewed offers is between 2 and 3 offers

In [ ]:
data[data['offer_type']=='discount'].groupby('customer_id')['offer-completed'].count()

In [ ]:
data[data['offer_type']=='discount'].groupby('customer_id')['offer-completed'].count().hist()
plt.title('Discount Offer Received by User');

##### Detection
* Yet again, the highest ratio of completed offers is between 2 and 3 offers

### Modeling and Predictions

In [ ]:
data['gender'].value_counts()

In [ ]:
genders = {'O': 0, 'M': 1, 'F': 2}
data['gender'] = data['gender'].map(genders)

In [ ]:
data['offer_type'].value_counts()

In [ ]:
offers = {'bogo': 0, 'discount': 1, 'informational': 2}
data['offer_type'] = data['offer_type'].map(offers)

In [ ]:
data.head(3)

In [ ]:
data.columns

In [ ]:
X = data.drop(columns=['customer_id', 'event_id' , 'event' , 'became_member_on','offer-completed', 'offer-received',
       'offer-viewed', 'transaction'])
Y = data['event_id']

In [ ]:
X.shape , Y.shape

In [ ]:
X.head()

In [ ]:
Y.head()

#### Feature Scaling

In [ ]:
class_name = ['offer recieved', 'offer viewed', 'transaction', 'offer completed']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
std = StandardScaler()

In [ ]:
X_train['income'] = std.fit_transform(X_train['income'].values.reshape(-1, 1))
X_train['age'] = std.fit_transform(X_train['age'].values.reshape(-1, 1))

X_train.reset_index(inplace=True)
X_train = X_train.drop(['index'], axis=1)

In [ ]:
X_test['income'] = std.transform(X_test['income'].values.reshape(-1, 1))
X_test['age'] = std.fit_transform(X_test['age'].values.reshape(-1, 1))

X_test.reset_index(inplace=True)
X_test = X_test.drop(['index'], axis=1)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
# converting to a numpy array
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

#### Building a model

In [ ]:
ann = keras.models.Sequential()

In [ ]:
ann.add(keras.layers.Dense(6, activation='relu'))
ann.add(keras.layers.Dense(6, activation='relu'))
ann.add(keras.layers.Dense(4, activation = 'softmax'))

In [ ]:
ann.compile(optimizer = 'adam', 
            loss = 'sparse_categorical_crossentropy', 
            metrics = ['accuracy'])

In [ ]:
ann_history = ann.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=15, batch_size=100)

In [ ]:
ann.summary()

In [ ]:
ann_history.params

In [ ]:
pd.DataFrame(ann.history.history).plot(figsize=(8,5))
plt.grid(True)
plt.gca().set_ylim(0.2,0.3)
plt.show()

#### Detection
* Evaluation can be made on the basis of the accuracy obtained by the model and by the loss produced

In [ ]:
ann.evaluate(X_test , y_test)

##### Detection:
* As shown above in the evaluation, the test accuracy is only ~25%
* This means that this model need a few corrections and improvements in order to deliver better results

### Refinement : Improving Prediction Model
* In this improvment process, I've deciede to recreate the X-DataFrame with highly recommended and dependent features

In [ ]:
# The following steps will look a lot like the previous one, with just a little bit of adjustments
data.columns

In [ ]:
data.groupby(['event'])['customer_id'].count()

In [ ]:
data = data.groupby('event').sample(n=30000)

In [ ]:
data.groupby(['event'])['customer_id'].count()

In [ ]:
X = data.drop(['customer_id', 'event_id' ,  'amount','event' , 'became_member_on','offer-completed', 'offer-received',
       'offer-viewed','email', 'mobile', 'social', 'web', 'time','transaction', 'duration'], axis=1)
Y = data['event_id']

In [ ]:
X.head()

In [ ]:
Y.head()

In [ ]:
X.shape , Y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
std = StandardScaler()

In [ ]:
X_train['income'] = std.fit_transform(X_train['income'].values.reshape(-1, 1))
X_train['age'] = std.fit_transform(X_train['age'].values.reshape(-1, 1))

X_train.reset_index(inplace=True)
X_train = X_train.drop(['index'], axis=1)

In [ ]:
X_test['income'] = std.transform(X_test['income'].values.reshape(-1, 1))
X_test['age'] = std.fit_transform(X_test['age'].values.reshape(-1, 1))

X_test.reset_index(inplace=True)
X_test = X_test.drop(['index'], axis=1)

In [ ]:
# converting to a numpy array
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

#### Build a model

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
accuracy_score(y_train,np.ones(len(y_train)))

In [ ]:
ann = keras.models.Sequential()

In [ ]:
ann.add(keras.layers.Dense(64, kernel_initializer = 'normal', activation='relu'))
ann.add(keras.layers.Dense(32, kernel_initializer = 'normal', activation='relu'))
ann.add(keras.layers.Dense(15, kernel_initializer = 'normal', activation='relu'))
ann.add(keras.layers.Dense(5, kernel_initializer = 'normal', activation='relu'))
ann.add(keras.layers.Dense(10, kernel_initializer = 'normal', activation='relu'))
ann.add(keras.layers.Dense(4, kernel_initializer = 'normal', activation = 'softmax'))

In [ ]:
ann.compile(optimizer = 'adam', 
            loss = 'sparse_categorical_crossentropy', 
            metrics = ['acc'])

In [ ]:
ann_history = ann.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=100)

In [ ]:
ann.summary()

In [ ]:
ann_history.params

In [ ]:
pd.DataFrame(ann.history.history).plot(figsize=(8,5))
plt.grid(True)
plt.gca().set_ylim(0.2,0.4)
plt.show()

In [ ]:
ann.evaluate(X_test , y_test)

In [ ]:
# TO PERFORM OF CHECK THE RESULT IN NEW DATA SET
# AS WE DON'T HAVE NEW DATA SET, CREATE ONE FROM TEST DATA SET
# HOW TO PREDICT THE PROBABILITY and CLASSES IN UNSEEN DATA


x_new = X_test[:3]

In [ ]:
# PROBABILITY OF EACH SET

y_prob = ann.predict(x_new)
y_prob.round(2)                        # RESULT IN 2 DECIMAL PLACE 

In [ ]:
# CLASS OF EACH SET 

y_pred = ann.predict_classes(x_new)
y_pred

In [ ]:
np.array(class_name)[y_pred]

##### Detection
* The results between the first and second model changed by a bit.
* In the last step above, we can see that the events were predicted wrongly as 'offer received'

### Conclusion
* I think it is obvious that this project was very challenging, due to the structure of the data
* As shown in the 'Detection' above, the predictions were wrongly analyzed as 'offer received'. This is due to the amount of occurences of the offer type.
* While majority classes were performing well, minority classes didn't perform well, due to the imbalanced dataset
* My personal goal was to build an analysis (something practical) for the Starbucks company in order to help them by making the choices more efficient

**Potentional improvment:** Analysing and building the deep learning models.

* The result seemed to improve by changing the model-logic. The accuracy increased by bit.
* After taking a sample amount of each event, we saw that the accuracy increased a bit.
* At this point one should consider to use more balanced datasets to ensure that the most possible accuracy is reached

#### References
* https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.replace.html
* https://pandas.pydata.org/pandas-docs/version/0.23.4/generated/pandas.merge.html
* https://stackoverflow.com/questions/37600711/pandas-split-column-into-multiple-columns-by-comma
* https://www.researchgate.net/post/Is_there_a_universal_method_rule_to_choose_the_activation_function_for_a_MLP_neural_network
* https://stackoverflow.com/questions/55324762/the-added-layer-must-be-an-instance-of-class-layer-found-tensorflow-python-ke
* https://www.analyticsvidhya.com/blog/2020/04/feature-scaling-machine-learning-normalization-standardization/
* https://www.tensorflow.org/tutorials/keras/classification
* https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
* https://machinelearningmastery.com/tutorial-first-neural-network-python-keras/